In [ ]:
import pandas as pd

home_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\raw 데이터\home_purchases_loanremove.csv"
)

print(f"행 수: {home_df.shape[0]:,}")
print(f"컬럼 수: {home_df.shape[1]:,}")

display(home_df.head())

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
region_cols = [
    "region",
    "residence_region",
    "lawd_cd",
    "lawd_name",
    "full_region_name",
    "umd_nm",
    "sido",
    "sigungu",
]

# 실제 존재하는 지역 컬럼만 선택
region_cols = [col for col in region_cols if col in home_df.columns]

print("지역 관련 컬럼:", region_cols)
#display(home_df[region_cols].head())

In [ ]:
display(home_df[region_cols].head().T)

In [ ]:
#지역관련 8개의 컬럼 삭제 나중에 지역통합버전과 합침.
#home_df = home_df.drop(columns=region_cols)

#print(f"행 수: {home_df.shape[0]:,}")
#print(f"컬럼 수: {home_df.shape[1]:,}")

In [ ]:
# 남은 108개의컬럼별 결측값·0 개수 확인
check_df = pd.DataFrame({
    "자료형": home_df.dtypes.astype(str),
    "결측 개수": home_df.isna().sum(),
    "0 개수": home_df.eq(0).sum(),
    "고유값 수": home_df.nunique()
})

check_df["결측 비율(%)"] = (
    check_df["결측 개수"] / len(home_df) * 100
)
check_df["0 비율(%)"] = (
    check_df["0 개수"] / len(home_df) * 100
)

# 결측값이나 0이 있는 컬럼만 표시
check_result = check_df[
    (check_df["결측 개수"] > 0) |
    (check_df["0 개수"] > 0)
].sort_values(
    ["결측 개수", "0 개수"],
    ascending=False
)

with pd.option_context("display.max_rows", None):
    display(check_result)

In [ ]:
#대출관련 결측치 확인중
loan_check_cols = [
    "loan_types",
    "loan_opened_at",
    "mortgage_rate",
    "mortgage_term_years",
    "ltv_at_origination",
    "monthly_payment_total",
    "annual_debt_service_total",
    "weighted_rate",
    "loan_to_income",
    "payment_to_income",
]

# 매입 대출 여부별 고객 수
print("매입 대출 여부별 건수 (0: 없음, 1: 있음)")
display(home_df["bought_with_loan"].value_counts().sort_index())

# 매입 대출 여부별 결측 개수
print("매입 대출 여부별 결측 개수")
display(
    home_df.groupby("bought_with_loan")[loan_check_cols]
    .agg(lambda col: col.isna().sum())
    .T
)

In [ ]:
# 매입 대출이 없는 사례
no_purchase_loan = home_df["bought_with_loan"].eq(0)

zero_fill_cols = [
    "monthly_payment_total",
    "annual_debt_service_total",
    "loan_to_income",
    "payment_to_income",
]

home_df.loc[no_purchase_loan, zero_fill_cols] = (
    home_df.loc[no_purchase_loan, zero_fill_cols].fillna(0)
)

# 처리 후 확인
display(home_df[zero_fill_cols].isna().sum())

In [ ]:
check_cols = [
    "customer_id",
    "bought_with_loan",
    "total_loan_principal",
    "monthly_payment_total",
    "annual_debt_service_total",
    "loan_to_income",
    "payment_to_income",
    "mortgage_rate",
    "mortgage_term_years",
]

# 각 그룹에서 2건씩 선택
no_loan = home_df[home_df["bought_with_loan"] == 0].head(2)
has_loan = home_df[home_df["bought_with_loan"] == 1].head(2)

# 세로로 펼쳐 비교
comparison = pd.concat([no_loan, has_loan])[check_cols].T
comparison.columns = [
    "매입 대출 없음①",
    "매입 대출 없음②",
    "매입 대출 있음①",
    "매입 대출 있음②",
]

display(comparison)

In [ ]:
# 매입 대출이 없는 사례만 확인
no_loan = home_df.loc[home_df["bought_with_loan"].eq(0)]

zero_cols = [
    "total_loan_principal",
    "mortgage_principal",
    "other_loan_principal",
    "monthly_payment_total",
    "annual_debt_service_total",
    "loan_to_income",
    "payment_to_income",
]

blank_cols = [
    "loan_types",
    "loan_opened_at",
    "mortgage_rate",
    "mortgage_term_years",
    "ltv_at_origination",
    "weighted_rate",
]

check = pd.DataFrame({
    "기대 상태": ["0"] * len(zero_cols) + ["빈칸"] * len(blank_cols),
    "0 개수": no_loan[zero_cols + blank_cols].eq(0).sum(),
    "빈칸 개수": no_loan[zero_cols + blank_cols].isna().sum(),
})

check["처리 확인"] = [
    no_loan[col].eq(0).all() for col in zero_cols
] + [
    no_loan[col].isna().all() for col in blank_cols
]

print(f"매입 대출 없는 사례: {len(no_loan):,}건")
display(check)

In [ ]:
# 건축연도가 없는 행 확인 — 데이터 변경 없음
missing_build_year = home_df.loc[
    home_df["build_year"].isna(),
    [
        "real_asset_id",
        "customer_id",
        "apt_name",
        "exclusive_area_m2",
        "floor",
        "build_year",
        "purchase_date",
        "purchase_reference_price",
    ]
]

display(missing_build_year)

In [ ]:
# 원본에서 확인에 필요한 컬럼만 읽기
building_check = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\raw 데이터\home_purchases_loanremove.csv",
    usecols=[
        "real_asset_id",
        "apt_name",
        "full_region_name",
        "build_year",
    ]
)

# 같은 이름의 아파트 확인
same_apt = building_check.loc[
    building_check["apt_name"].eq("더샵원주센트럴파크1단지")
]

display(same_apt)

print("건축연도별 건수:")
display(same_apt["build_year"].value_counts(dropna=False))

In [ ]:
# 같은 단지에서 건축연도가 비어 있는 부동산 ID
missing_ids = same_apt.loc[
    same_apt["full_region_name"].eq("강원특별자치도 원주시 무실동")
    & same_apt["build_year"].isna(),
    "real_asset_id"
]

# 작업 데이터에서 해당 3건의 빈칸만 2021로 채우기
fill_mask = (
    home_df["real_asset_id"].isin(missing_ids)
    & home_df["build_year"].isna()
)

print("수정 대상:", fill_mask.sum(), "건")

home_df.loc[fill_mask, "build_year"] = 2021

# 처리 결과 확인
display(
    home_df.loc[
        home_df["real_asset_id"].isin(missing_ids),
        ["real_asset_id", "apt_name", "build_year"]
    ]
)

print("남은 건축연도 결측:", home_df["build_year"].isna().sum(), "건")

In [ ]:
# 숫자형 컬럼만 대상으로 확인
numeric_df = home_df.select_dtypes(include="number")

zero_check = pd.DataFrame({
    "0 개수": numeric_df.eq(0).sum(),
    "0 비율(%)": numeric_df.eq(0).sum() / len(home_df) * 100,
    "결측 개수": numeric_df.isna().sum(),
    "고유값 수": numeric_df.nunique(),
})

# 0이 있는 컬럼을 비율이 높은 순서로 표시
zero_check = zero_check.loc[
    zero_check["0 개수"] > 0
].sort_values("0 비율(%)", ascending=False)

with pd.option_context("display.max_rows", None):
    display(zero_check)

0많은 컬럼 삭제필요없음.

In [ ]:
#자기자금 0인 2건 확인
display(
    home_df.loc[
        home_df["own_funds_est"].eq(0),
        [
            "customer_id",
            "real_asset_id",
            "purchase_price_est",
            "total_loan_principal",
            "mortgage_principal",
            "other_loan_principal",
            "own_funds_est",
            "own_funds_ratio",
        ]
    ]
)

자기자금 = 구매가격 − 총 매입 대출   
자기자금이 구매가격을 초과함.  


In [ ]:
# 확인만 진행 — 데이터 변경 없음
exceed_mask = (
    home_df["total_loan_principal"]
    > home_df["purchase_price_est"]
)

print("대출금이 구매가격을 초과한 사례:", exceed_mask.sum(), "건")

In [ ]:
# 1. 높은 대출 상환 부담 플래그와 DSR 비교
print("대출 상환 부담 플래그별 DSR 범위")
display(
    home_df.groupby("high_debt_service_flag")["dsr_income"]
    .agg(["count", "min", "max"])
)

# 2. 높은 주거·교육 고정 부담 플래그와 부담 비율 비교
print("고정 부담 플래그별 부담 비율 범위")
display(
    home_df.groupby("high_fixed_burden_flag")[
        "housing_education_burden_ratio"
    ]
    .agg(["count", "min", "max"])
)

In [ ]:
checks = [
    ("high_debt_service_flag", "dsr_income", 0.4),
    ("high_fixed_burden_flag", "housing_education_burden_ratio", 0.5),
]

for flag_col, ratio_col, threshold in checks:
    actual = home_df[flag_col]

    print(f"\n{flag_col}")
    print(
        f"{threshold:.0%} 초과 기준과 다른 행:",
        actual.ne((home_df[ratio_col] > threshold).astype(int)).sum()
    )
    print(
        f"{threshold:.0%} 이상 기준과 다른 행:",
        actual.ne((home_df[ratio_col] >= threshold).astype(int)).sum()
    )

In [ ]:
from itertools import combinations

duplicate_pairs = []

for col1, col2 in combinations(home_df.columns, 2):
    a = home_df[col1]
    b = home_df[col2]

    # 행별 값 비교: 둘 다 결측이면 같은 값으로 취급
    same = (
        a.eq(b).fillna(False)
        | (a.isna() & b.isna())
    )

    if same.all():
        duplicate_pairs.append({
            "컬럼 1": col1,
            "컬럼 2": col2,
            "둘 다 전체 결측": a.isna().all() and b.isna().all(),
        })

duplicate_columns = pd.DataFrame(
    duplicate_pairs,
    columns=["컬럼 1", "컬럼 2", "둘 다 전체 결측"]
)

print(f"값이 완전히 같은 컬럼 쌍: {len(duplicate_columns)}개")

with pd.option_context("display.max_rows", None):
    display(duplicate_columns)

In [ ]:
drop_duplicate_cols = [
    "asset_name",
    "purchase_price_est",
    "debt_is_home_owner",
    "debt_monthly_private_education_cost",
]

home_df = home_df.drop(columns=drop_duplicate_cols)

print(f"행 수: {home_df.shape[0]:,}")
print(f"컬럼 수: {home_df.shape[1]:,}")

중복 컬럼 정리 6쌍  
- 삭제: `asset_name`, `purchase_price_est`, `debt_is_home_owner`, `debt_monthly_private_education_cost`
- 유지: 각 삭제 컬럼에 대응하는 `apt_name`, `purchase_reference_price`, `is_home_owner`, `monthly_private_education_cost`
- 의미가 달라 둘 다 유지: `estimated_value` / `home_value`, `purchase_date` / `reference_deal_date`

In [ ]:
# 결측값도 하나의 값으로 포함해서 확인
unique_counts = home_df.nunique(dropna=False)
constant_cols = unique_counts[unique_counts == 1].index

constant_check = pd.DataFrame([
    {
        "컬럼": col,
        "공통 값": home_df[col].iloc[0],
    }
    for col in constant_cols
], columns=["컬럼", "공통 값"])

display(constant_check)

In [ ]:
#단일값 컬럼 제외|
#drop_constant_cols = ["asset_type","created_at","price_source","is_home_owner","cf_months",]
#home_df = home_df.drop(columns=drop_constant_cols)

print(f"행 수: {home_df.shape[0]:,}")
print(f"컬럼 수: {home_df.shape[1]:,}")

In [ ]:
print("완전히 동일한 중복 행:", home_df.duplicated().sum(), "건")
print("부동산 ID 중복:", home_df["real_asset_id"].duplicated().sum(), "건")
print("고객 ID 중복:", home_df["customer_id"].duplicated().sum(), "건")

In [ ]:
print("문자형 컬럼 목록")
display(
    home_df.select_dtypes(include=["object", "string"])
    .dtypes.rename("자료형")
)

# 날짜로 해석되지 않는 값 확인
date_cols = ["purchase_date", "reference_deal_date", "loan_opened_at"]

date_check = []

for col in date_cols:
    converted = pd.to_datetime(home_df[col], errors="coerce")

    date_check.append({
        "컬럼": col,
        "원래 빈칸": home_df[col].isna().sum(),
        "값은 있지만 날짜 변환 실패": (
            home_df[col].notna() & converted.isna()
        ).sum(),
    })

display(pd.DataFrame(date_check))

In [ ]:
# 비교용 숫자 변환 — home_df는 변경하지 않음
def num(col):
    return pd.to_numeric(home_df[col], errors="coerce")

checks = {
    "참고 구매가격이 0 이하": num("purchase_reference_price") <= 0,
    "부동산 평가액이 0 이하": num("estimated_value") <= 0,
    "전용면적이 0 이하": num("exclusive_area_m2") <= 0,
    "전체 매입 대출금이 음수": num("total_loan_principal") < 0,
    "주담대 최초 원금이 음수": num("mortgage_principal") < 0,
    "기타 매입 대출금이 음수": num("other_loan_principal") < 0,
    "개인 연소득이 음수": num("primary_annual_income") < 0,
    "가구 연소득이 0 이하": num("household_annual_income") <= 0,
    "월 매입 대출 상환액이 음수": num("monthly_payment_total") < 0,
    "금리가 0 이하 — 확인 필요": num("mortgage_rate") <= 0,
    "대출 기간이 0 이하": num("mortgage_term_years") <= 0,
    "건축연도가 0 이하": num("build_year") <= 0,
    "건축연도가 현재 연도보다 미래": num("build_year") > 2026,
    "매입 나이가 음수": num("age_at_purchase") < 0,
}

range_check = pd.DataFrame({
    "확인 항목": checks.keys(),
    "해당 건수": [mask.sum() for mask in checks.values()],
})

display(range_check)

In [ ]:
date_cols = [
    "purchase_date",
    "reference_deal_date",
    "loan_opened_at",
]

for col in date_cols:
    home_df[col] = pd.to_datetime(home_df[col], errors="raise")

display(home_df[date_cols].dtypes)

display(home_df[date_cols].agg(["min", "max"]))

# 데이터 기준월의 마지막 날
reference_date = pd.Timestamp("2026-09-30")

print(
    "기준월 이후 구매 사례:",
    home_df["purchase_date"].gt(reference_date).sum(),
    "건"
)

구매일은 2018년 10월 1일~2026년 9월 11일

In [ ]:
# 원본에서 고객별 생년월일 가져오기
birth_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\full data\customer_financial_profiles_202610061750.csv",
    usecols=["customer_id", "birth_date"]
)

birth_map = birth_df.set_index("customer_id")["birth_date"]
birth = pd.to_datetime(home_df["customer_id"].map(birth_map))
purchase = home_df["purchase_date"]

# 매입일에 생일이 지나지 않았으면 1살 차감
before_birthday = (
    (purchase.dt.month < birth.dt.month)
    | (
        (purchase.dt.month == birth.dt.month)
        & (purchase.dt.day < birth.dt.day)
    )
)

exact_age = (
    purchase.dt.year
    - birth.dt.year
    - before_birthday.astype(int)
)

age_check = pd.DataFrame({
    "customer_id": home_df["customer_id"],
    "생년월일": birth,
    "매입일": purchase,
    "기존 매입 나이": home_df["age_at_purchase"],
    "생일 기준 만 나이": exact_age,
})

different = age_check["기존 매입 나이"].ne(exact_age)

print("생년월일 연결 실패:", birth.isna().sum(), "건")
print("기존 나이와 다른 사례:", different.sum(), "건")
print("정확한 만 나이 기준 20세 미만:", exact_age.lt(20).sum(), "건")

display(age_check.loc[different].head(10))

In [ ]:
# 매입 나이를 정확한 만 나이로 변경
home_df["age_at_purchase"] = exact_age.astype(int)

# 수정된 나이로 20세 미만 여부 재계산
home_df["under20_at_purchase"] = (
    home_df["age_at_purchase"] < 20
).astype(int)

print(
    "기존 나이와 달라 수정한 사례:",
    different.sum(), "건"
)

print(
    "20세 미만 매입 사례:",
    home_df["under20_at_purchase"].sum(), "건"
)

In [ ]:
# 수정 전 나이 보관 — 확인용 변수
age_before = home_df["age_at_purchase"].copy()

# 생일 기준 만 나이로 수정
home_df["age_at_purchase"] = exact_age.astype(int)

# 20세 미만 여부 재계산
home_df["under20_at_purchase"] = (
    home_df["age_at_purchase"] < 20
).astype(int)

print(
    "나이가 변경된 사례:",
    age_before.ne(home_df["age_at_purchase"]).sum(), "건"
)
print(
    "20세 미만 매입 사례:",
    home_df["under20_at_purchase"].sum(), "건"
)
print("전체 행 수:", len(home_df))

In [ ]:
young = home_df.loc[home_df["age_at_purchase"] < 20]

print("20세 미만 매입 사례:", len(young), "건")

# 나이별 건수
display(
    young["age_at_purchase"]
    .value_counts()
    .sort_index()
    .rename_axis("매입 당시 만 나이")
    .to_frame("건수")
)

# 매입 대출 유무별 건수
display(
    young.groupby("bought_with_loan")
    .size()
    .rename_axis("매입 대출 여부 (0: 없음, 1: 있음)")
    .to_frame("건수")
)

#이부분부터 회의때 논의

In [ ]:
young_check = pd.crosstab(
    young["age_at_purchase"],
    young["bought_with_loan"]
).reindex(columns=[0, 1], fill_value=0)

young_check.columns = ["매입 대출 없음", "매입 대출 있음"]
young_check.index.name = "매입 당시 만 나이"

display(young_check)

In [ ]:
import numpy as np

loan_sum = (
    home_df["mortgage_principal"]
    + home_df["other_loan_principal"]
)

same = np.isclose(
    home_df["total_loan_principal"],
    loan_sum,
    rtol=0,
    atol=0.01  # 소수점 계산 오차 0.01원까지 허용
)

print("대출금 합계 불일치:", (~same).sum(), "건")

In [ ]:
# 매입 대출금 ÷ 가구 연소득
expected_loan_ratio = (
    home_df["total_loan_principal"]
    / home_df["household_annual_income"]
)

# 매입 대출 연 상환액 ÷ 가구 연소득
expected_payment_ratio = (
    home_df["annual_debt_service_total"]
    / home_df["household_annual_income"]
)

loan_match = np.isclose(
    home_df["loan_to_income"],
    expected_loan_ratio,
    rtol=1e-6,
    atol=1e-8
)

payment_match = np.isclose(
    home_df["payment_to_income"],
    expected_payment_ratio,
    rtol=1e-6,
    atol=1e-8
)

print("소득 대비 매입 대출금 비율 불일치:", (~loan_match).sum(), "건")
print("소득 대비 매입 대출 연 상환액 비율 불일치:", (~payment_match).sum(), "건")

In [ ]:
checks = {
    "loan_to_income": (
        home_df["total_loan_principal"]
        / home_df["household_annual_income"]
    ),
    "payment_to_income": (
        home_df["annual_debt_service_total"]
        / home_df["household_annual_income"]
    ),
}

for col, expected in checks.items():
    match = np.isclose(
        home_df[col],
        expected,
        rtol=1e-6,
        atol=1e-8
    )

    print(f"{col} 계산 불일치: {(~match).sum()}건")

In [ ]:
check_cols = [
    "purchase_reference_price",  # 참고 구매가격
    "household_annual_income",    # 가구 연소득
    "total_loan_principal",       # 매입 대출금
    "monthly_payment_total",     # 매입 대출 월 상환액
    "loan_to_income",            # 가구 연소득 대비 매입 대출금
    "payment_to_income",         # 가구 연소득 대비 매입 대출 연 상환액
    "dsr_income",                # 현재 소득 대비 부채 상환 부담
    "own_funds_ratio",           # 구매가격 대비 추정 자기자금 비율
]

display(
    home_df[check_cols]
    .describe(percentiles=[0.5, 0.95, 0.99])
    .T[["min", "50%", "95%", "99%", "max"]]
)

In [ ]:
display(
    home_df.nlargest(10, "payment_to_income")[
        [
            "customer_id",
            "age_at_purchase",
            "purchase_reference_price",
            "household_annual_income",
            "total_loan_principal",
            "monthly_payment_total",
            "annual_debt_service_total",
            "payment_to_income",
            "dsr_income",
        ]
    ]
)

In [ ]:

# 기존 자기자금 계산 검증
expected_funds = (
    home_df["purchase_reference_price"]
    - home_df["total_loan_principal"]
).clip(lower=0)

expected_ratio = (
    expected_funds / home_df["purchase_reference_price"]
)

checks = {
    "자기자금 계산 불일치": (~np.isclose(
        home_df["own_funds_est"], expected_funds,
        rtol=1e-6, atol=0.01
    )).sum(),

    "자기자금 비율 계산 불일치": (~np.isclose(
        home_df["own_funds_ratio"], expected_ratio,
        rtol=1e-6, atol=1e-8
    )).sum(),

    "전체 중복 행": home_df.duplicated().sum(),
    "부동산 ID 중복": home_df["real_asset_id"].duplicated().sum(),
}

display(pd.Series(checks, name="건수").to_frame())

# 의도적으로 유지한 대출 비해당 결측
allowed_missing = [
    "loan_types",
    "loan_opened_at",
    "mortgage_rate",
    "mortgage_term_years",
    "ltv_at_origination",
    "weighted_rate",
]

missing = home_df.isna().sum()
missing = missing[missing > 0]

print("남아 있는 결측값")
display(missing.rename("결측 개수").to_frame())

unexpected = missing.drop(labels=allowed_missing, errors="ignore")
print("예상하지 못한 결측 컬럼 수:", len(unexpected))

print(f"최종 크기: {len(home_df):,}행 × {home_df.shape[1]}개 컬럼")

print(
    "매입 나이 20세 미만 — 판단 보류:",
    home_df["under20_at_purchase"].sum(), "건"
)

print(
    "매입 대출금이 구매가격 초과 — 판단 보류:",
    (
        home_df["total_loan_principal"]
        > home_df["purchase_reference_price"]
    ).sum(), "건"
)

In [ ]:
from pathlib import Path

output_path = Path(
    r"C:\ai_campus\미니프로젝트_마이데이터"
    r"\csv\re전처리\home_purchases_전처리최종.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

home_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

print(f"저장 크기: {len(home_df):,}행 × {home_df.shape[1]}개 컬럼")
print("저장 완료:", output_path)